# VideoSearch Unified Runner

Kurumun kendi video/CSV datasetini hazırlamak, CSV kolonlarını FAZ11 canonical şemasına eşlemek, production manifest/preflight koduyla doğrulamak ve uygun ortamda embedding veya Docker ingest çalıştırmak için tek giriş notebook'u.

### Modlar

- `prepare_dataset`: GPU/Docker gerekmeden profiling, mapping, manifest ve onboarding bundle.
- `portable_embedding`: Colab veya yerel NVIDIA GPU'da Qwen window embedding.
- `docker_ingest`: Notebook ile Docker FAZ11 aynı makinedeyse `ingest --resume`.

### Veri kaynakları

- `local_path`
- `zip_upload`
- `google_drive` — yalnız seçildiğinde mount edilir
- `remote_url`
- `sample_dataset`

> Colab ile uzak kurum Docker sunucusu aynı dosya sistemini paylaşmaz.

## 1. Ayarlar — yalnız bu hücreyi düzenleyin

In [ ]:
MODE = "prepare_dataset"  # prepare_dataset | portable_embedding | docker_ingest

DATA_SOURCE = "sample_dataset"  # local_path | zip_upload | google_drive | remote_url | sample_dataset
LOCAL_DATA_ROOT = ""
ZIP_PATH = ""
DRIVE_DATA_ROOT = "/content/drive/MyDrive/Multimodal-Video-Intelligence/data"
REMOTE_URL = ""
REMOTE_SHA256 = ""

REPO_URL = "https://github.com/ColdVI/Multimodal-Video-Intelligence.git"
REPO_REF = "main"
REPO_ROOT_OVERRIDE = ""

DATASET_ID = "institution_dataset"
DISPLAY_NAME = "Institution Dataset"
VIDEO_GLOB = "videos/**/*.mp4"
TELEMETRY_GLOB = "telemetry/**/*.csv"
PAIRING_STRATEGY = "filename_stem"
PAIRING_CSV = "pairing.csv"

WINDOW_SIZE_S = 8.0
STRIDE_S = 4.0
FRAMES_PER_ITEM = 8
PARTIAL_WINDOW_POLICY = "drop_partial"

TELEMETRY_CLOCK = "relative_s"  # relative_s | unix_s | unix_ms | iso8601
TIMESTAMP_COLUMN = "timestamp"
VIDEO_START_TIME_FROM = None     # container_creation_time | filename | manifest_csv
TIMEZONE = "UTC"
OFFSET_S = 0.0
MAX_GAP_S = 1.0

CANONICAL_FIELDS = {
    # "altitude_m": {
    #   "source": "RelAltitude", "unit": "m", "reference": "AGL",
    #   "type": "continuous", "interpolation": "linear", "aggregation": "median",
    # },
}
EXTRA_FIELDS = {
    # "battery_v": {"source": "BatteryVoltage", "unit": "V", "type": "continuous"},
}

MODEL_ID = "Qwen/Qwen3-VL-Embedding-2B"
MODEL_REVISION = "9f2f7e710d6d81056aa5c0a4f04764fec6bb7bda"
SOURCE_REPO = "https://github.com/QwenLM/Qwen3-VL-Embedding.git"
SOURCE_COMMIT = "393e2978d27852b0d0230d6994f37f9c15bed73c"
ATTN_IMPL = "sdpa"
EMBED_BATCH_SIZE = 2
SHARD_WINDOWS = 64
MRL_DIMENSIONS = (2048, 1024, 512, 256)
MAX_WINDOWS = None

ENV_FILE = ".env"
START_DOCKER_SERVICES = False
RUN_DOCKER_INGEST = False

print("MODE:", MODE, "| DATA_SOURCE:", DATA_SOURCE)

## 2. Ortam ve repository

In [ ]:
from __future__ import annotations
import csv, hashlib, json, os, platform, shutil, subprocess, sys, urllib.request, zipfile
from datetime import datetime, timezone
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
BASE_ROOT = Path("/content") if IS_COLAB else Path.cwd()
WORKSPACE = BASE_ROOT / "mvi_unified_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)

def run(command, *, cwd=None, check=True):
    command = [str(x) for x in command]
    print("+", " ".join(command))
    return subprocess.run(command, cwd=cwd, check=check, text=True)

def locate_repo() -> Path:
    if REPO_ROOT_OVERRIDE:
        candidate = Path(REPO_ROOT_OVERRIDE).expanduser().resolve()
        if not (candidate / "service" / "app").is_dir():
            raise FileNotFoundError(candidate)
        return candidate
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "service" / "app").is_dir():
            return candidate
    target = WORKSPACE / "Multimodal-Video-Intelligence"
    if not target.is_dir():
        run(["git", "clone", REPO_URL, target])
    run(["git", "fetch", "--all", "--tags"], cwd=target)
    run(["git", "checkout", REPO_REF], cwd=target)
    return target.resolve()

REPO_ROOT = locate_repo()
SERVICE_ROOT = REPO_ROOT / "service"
if str(SERVICE_ROOT) not in sys.path:
    sys.path.insert(0, str(SERVICE_ROOT))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Colab:", IS_COLAB)
print("Repository:", REPO_ROOT)
print("Workspace:", WORKSPACE)

## 3. Veri kaynağını hazırla

In [ ]:
VIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".avi"}

def sha256_file(path: Path, chunk_size=1024*1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def safe_extract_zip(zip_path: Path, destination: Path) -> Path:
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            member = Path(info.filename.replace("\\", "/"))
            if member.is_absolute() or ".." in member.parts:
                raise ValueError(f"Güvensiz ZIP üyesi: {info.filename}")
            if not (root / member).resolve().is_relative_to(root):
                raise ValueError(f"ZIP path traversal: {info.filename}")
        archive.extractall(root)
    children = [p for p in root.iterdir() if p.name != "__MACOSX"]
    return children[0] if len(children) == 1 and children[0].is_dir() else root

def download_with_resume(url: str, output: Path) -> Path:
    output.parent.mkdir(parents=True, exist_ok=True)
    existing = output.stat().st_size if output.exists() else 0
    request = urllib.request.Request(url, headers={"Range": f"bytes={existing}-"} if existing else {})
    with urllib.request.urlopen(request, timeout=60) as response:
        mode = "ab" if existing and getattr(response, "status", 200) == 206 else "wb"
        with output.open(mode) as handle:
            shutil.copyfileobj(response, handle)
    return output

def create_sample_dataset(root: Path) -> Path:
    root = root.resolve()
    (root / "videos").mkdir(parents=True, exist_ok=True)
    (root / "telemetry").mkdir(parents=True, exist_ok=True)
    video = root / "videos" / "sample_flight.mp4"
    if not video.exists():
        ffmpeg = shutil.which("ffmpeg")
        if not ffmpeg:
            raise RuntimeError("sample_dataset için ffmpeg gerekli.")
        run([ffmpeg, "-y", "-f", "lavfi", "-i",
             "testsrc=size=320x240:rate=10:duration=12",
             "-pix_fmt", "yuv420p", video])
    telemetry = root / "telemetry" / "sample_flight.csv"
    if not telemetry.exists():
        with telemetry.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.writer(handle)
            writer.writerow(["timestamp", "RelAltitude", "GroundSpd", "Heading", "BatteryVoltage"])
            for i in range(121):
                writer.writerow([i/10, 80+i*.1, 12+i*.01, (350+i)%360, 24-i*.005])
    return root

if DATA_SOURCE == "local_path":
    if not LOCAL_DATA_ROOT: raise ValueError("LOCAL_DATA_ROOT gerekli.")
    DATA_ROOT = Path(LOCAL_DATA_ROOT).expanduser().resolve()
elif DATA_SOURCE == "google_drive":
    if not IS_COLAB: raise RuntimeError("Yerel Jupyter için local_path kullanın.")
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = Path(DRIVE_DATA_ROOT).resolve()
elif DATA_SOURCE == "zip_upload":
    if IS_COLAB and not ZIP_PATH:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1: raise ValueError("Tek ZIP yükleyin.")
        name, payload = next(iter(uploaded.items()))
        local_zip = WORKSPACE / "uploads" / name
        local_zip.parent.mkdir(parents=True, exist_ok=True)
        local_zip.write_bytes(payload)
    else:
        if not ZIP_PATH: raise ValueError("ZIP_PATH gerekli.")
        local_zip = Path(ZIP_PATH).expanduser().resolve()
    DATA_ROOT = safe_extract_zip(local_zip, WORKSPACE / "extracted_dataset")
elif DATA_SOURCE == "remote_url":
    if not REMOTE_URL: raise ValueError("REMOTE_URL gerekli.")
    local_zip = download_with_resume(REMOTE_URL, WORKSPACE / "downloads" / "dataset.zip")
    if REMOTE_SHA256 and sha256_file(local_zip) != REMOTE_SHA256.lower():
        raise ValueError("REMOTE_SHA256 uyuşmuyor.")
    DATA_ROOT = safe_extract_zip(local_zip, WORKSPACE / "remote_dataset")
elif DATA_SOURCE == "sample_dataset":
    DATA_ROOT = create_sample_dataset(WORKSPACE / "sample_dataset")
    TIMESTAMP_COLUMN = "timestamp"
    VIDEO_GLOB = "videos/**/*.mp4"
    TELEMETRY_GLOB = "telemetry/**/*.csv"
    CANONICAL_FIELDS = {
        "altitude_m": {"source":"RelAltitude","unit":"m","reference":"AGL","type":"continuous",
                       "interpolation":"linear","aggregation":"median"},
        "velocity_mps": {"source":"GroundSpd","unit":"m/s","kind":"ground_speed","type":"continuous",
                         "interpolation":"linear","aggregation":"median"},
        "compass_heading": {"source":"Heading","unit":"deg","type":"circular_deg",
                            "interpolation":"circular","aggregation":"circular_mean"},
    }
    EXTRA_FIELDS = {"battery_v":{"source":"BatteryVoltage","unit":"V","type":"continuous"}}
else:
    raise ValueError(DATA_SOURCE)

if not DATA_ROOT.is_dir(): raise FileNotFoundError(DATA_ROOT)
print("DATA_ROOT:", DATA_ROOT)

## 4. Dataset ve CSV profiling

In [ ]:
import pandas as pd

def detect_csv(path: Path):
    for encoding in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            sample = path.read_text(encoding=encoding, errors="strict")[:32768]
            try:
                delimiter = csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
            except csv.Error:
                delimiter = ","
            frame = pd.read_csv(path, sep=delimiter, encoding=encoding, nrows=1000)
            return encoding, delimiter, frame
        except Exception:
            continue
    raise ValueError(f"CSV okunamadı: {path}")

video_paths = sorted(p for p in DATA_ROOT.rglob("*") if p.suffix.lower() in VIDEO_EXTENSIONS)
csv_paths = sorted(DATA_ROOT.rglob("*.csv"))
profile = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "data_root": str(DATA_ROOT),
    "video_count": len(video_paths),
    "telemetry_csv_count": len(csv_paths),
    "video_examples": [str(p.relative_to(DATA_ROOT)) for p in video_paths[:10]],
    "csv_files": [],
}
print("Video:", len(video_paths), "| CSV:", len(csv_paths))

first_frame = None
for path in csv_paths[:10]:
    encoding, delimiter, frame = detect_csv(path)
    item = {
        "path": str(path.relative_to(DATA_ROOT)),
        "encoding": encoding,
        "delimiter": delimiter,
        "columns": list(frame.columns),
        "rows_sampled": len(frame),
        "null_fraction": {c: float(frame[c].isna().mean()) for c in frame.columns},
        "dtypes": {c: str(t) for c, t in frame.dtypes.items()},
    }
    profile["csv_files"].append(item)
    first_frame = first_frame if first_frame is not None else frame
    print("\nCSV:", item["path"], "| delimiter:", repr(delimiter), "| encoding:", encoding)
    display(frame.head())

profile_path = WORKSPACE / "schema_profile.json"
profile_path.write_text(json.dumps(profile, ensure_ascii=False, indent=2), encoding="utf-8")
print("Profil:", profile_path)

if first_frame is not None:
    norm = {str(c).lower().replace("_","").replace("-",""): str(c) for c in first_frame.columns}
    aliases = {
        "timestamp": ("timestamp","time","flighttime","timeus","timems"),
        "altitude_m": ("relaltitude","altitude","altitudem","altaglm"),
        "velocity_mps": ("groundspd","groundspeed","velocity","speed"),
        "compass_heading": ("heading","headingdeg","compassheading","hdg"),
        "latitude": ("latitude","lat"),
        "longitude": ("longitude","lon","lng"),
    }
    print("\nMapping önerileri:")
    for target, candidates in aliases.items():
        match = next((norm[c] for c in candidates if c in norm), None)
        if match: print(f"  {target:18s} <- {match}")

## 5. Mapping kontrolü

Profil çıktısına göre ilk ayar hücresindeki `TIMESTAMP_COLUMN`, `CANONICAL_FIELDS` ve `EXTRA_FIELDS` değerlerini düzenleyin. Otomatik öneri yalnız öneridir.

In [ ]:
if csv_paths:
    _, _, sample_frame = detect_csv(csv_paths[0])
    available = set(map(str, sample_frame.columns))
    required = {spec["source"] for spec in [*CANONICAL_FIELDS.values(), *EXTRA_FIELDS.values()]}
    missing = sorted(required - available)
    if TIMESTAMP_COLUMN not in available:
        raise ValueError(f"Timestamp kolonu yok: {TIMESTAMP_COLUMN!r}; mevcut={sorted(available)}")
    if missing:
        raise ValueError(f"Mapped kolonlar CSV'de yok: {missing}")
    print("Mapping kolon kontrolü: PASS")

## 6. Production manifest üretimi ve parser doğrulaması

In [ ]:
import yaml

manifest_dir = REPO_ROOT / "datasets" / "generated"
manifest_dir.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = manifest_dir / f"{DATASET_ID}.yaml"

if TELEMETRY_CLOCK in {"unix_s","unix_ms","iso8601"} and VIDEO_START_TIME_FROM not in {
    "container_creation_time","filename","manifest_csv"
}:
    raise ValueError("Absolute telemetry için VIDEO_START_TIME_FROM zorunludur.")

manifest = {
    "schema_version": 1,
    "dataset_id": DATASET_ID,
    "display_name": DISPLAY_NAME,
    "source": {"videos_glob": VIDEO_GLOB, "video_id_from": "filename_stem"},
    "pairing": {
        "strategy": PAIRING_STRATEGY,
        "telemetry_glob": TELEMETRY_GLOB if csv_paths else None,
        "telemetry_id_from": "filename_stem",
        "manifest_csv": PAIRING_CSV if PAIRING_STRATEGY == "manifest_csv" else None,
    },
    "time_alignment": {
        "video_clock": "pts",
        "telemetry_clock": TELEMETRY_CLOCK,
        "video_start_time_from": VIDEO_START_TIME_FROM,
        "filename_time_regex": None,
        "filename_time_format": None,
        "timezone": TIMEZONE,
        "offset_s": float(OFFSET_S),
        "max_gap_s": float(MAX_GAP_S),
        "missing_policy": None,
    },
    "window": {
        "size_s": float(WINDOW_SIZE_S),
        "stride_s": float(STRIDE_S),
        "frames_per_item": int(FRAMES_PER_ITEM),
        "partial_window_policy": PARTIAL_WINDOW_POLICY,
    },
    "telemetry": {
        "format": "generic_csv",
        "timestamp_column": TIMESTAMP_COLUMN,
        "fields": CANONICAL_FIELDS,
        "extra": EXTRA_FIELDS,
    },
    "captions": {"format":"none"},
    "media": {"enabled":True,"clip_cache":True},
    "policy": {"fail_on_video_error":True},
}
MANIFEST_PATH.write_text(yaml.safe_dump(manifest, sort_keys=False, allow_unicode=True), encoding="utf-8")

from app.ingestion.manifest import load_manifest
parsed_manifest = load_manifest(MANIFEST_PATH)
print("Manifest parser: PASS")
print("Manifest:", MANIFEST_PATH)
print(MANIFEST_PATH.read_text(encoding="utf-8"))

## 7. Production data preflight

In [ ]:
from app.config import Settings
from app.preflight import run_data_preflight

artifact_root = WORKSPACE / "artifacts"
artifact_root.mkdir(parents=True, exist_ok=True)
env = dict(os.environ)
env.update({
    "EMBEDDING_MODE": "synthetic" if MODE == "prepare_dataset" else "real",
    "DATA_ROOT": str(DATA_ROOT),
    "ARTIFACTS_ROOT": str(artifact_root),
    "ENABLED_VECTOR_BACKENDS": "clickhouse",
    "ENABLED_DIMENSIONS": "512",
    "FILTER_EXECUTION_MODE": "pushdown",
    "MODEL_BUNDLE_ROOT": str(WORKSPACE / "model_bundle"),
})
configured = Settings.from_env(env)
preflight_report = run_data_preflight(MANIFEST_PATH, data_root=DATA_ROOT, configured=configured)

PREFLIGHT_PATH = WORKSPACE / "preflight.json"
PREFLIGHT_PATH.write_text(json.dumps(preflight_report, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

for item in preflight_report["checks"]:
    print(f"[{item['status'].upper():7s}] {item['check_id']}: {item['detail']}")

data_failures = [x for x in preflight_report["checks"] if x["category"]=="data" and x["status"]=="fail"]
if data_failures:
    raise RuntimeError(data_failures)
print("Data preflight: PASS | genel status:", preflight_report["status"])

## 8A. Dataset onboarding bundle

In [ ]:
bundle_root = WORKSPACE / "dataset_onboarding_bundle"
bundle_root.mkdir(parents=True, exist_ok=True)
shutil.copy2(MANIFEST_PATH, bundle_root / "dataset.yaml")
shutil.copy2(profile_path, bundle_root / "schema_profile.json")
shutil.copy2(PREFLIGHT_PATH, bundle_root / "preflight.json")
checksums = {p.name: sha256_file(p) for p in bundle_root.iterdir() if p.is_file()}
(bundle_root / "checksums.json").write_text(json.dumps(checksums, indent=2), encoding="utf-8")
(bundle_root / "README.txt").write_text(
    "Bu bundle ham video/telemetry içermez. dataset.yaml production parser ile doğrulanmıştır.\n",
    encoding="utf-8",
)
archive_path = shutil.make_archive(str(WORKSPACE / f"{DATASET_ID}_onboarding_bundle"), "zip", bundle_root)
print("Bundle:", archive_path)

## 8B. Portable Qwen embedding

Yalnız `MODE == "portable_embedding"` olduğunda çalışır. CUDA zorunludur. Production window generator ve MRL fonksiyonu kullanılır. Bu yol ClickHouse latency benchmark'ı veya kalıcı full-stack deployment değildir.

In [ ]:
if MODE != "portable_embedding":
    print("Atlandı: MODE portable_embedding değil.")
else:
    import numpy as np
    import pandas as pd
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("portable_embedding için CUDA GPU gereklidir.")

    run([sys.executable, "-m", "pip", "install", "-q",
         "transformers>=4.57.3", "accelerate>=1.12.0",
         "qwen-vl-utils>=0.0.14", "huggingface-hub", "pyarrow"])

    source_dir = WORKSPACE / "qwen_source"
    if not source_dir.is_dir():
        run(["git", "clone", SOURCE_REPO, source_dir])
    run(["git", "fetch", "--all"], cwd=source_dir)
    run(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=source_dir)
    observed = subprocess.check_output(["git","rev-parse","HEAD"], cwd=source_dir, text=True).strip()
    if observed != SOURCE_COMMIT:
        raise RuntimeError("Qwen source commit uyuşmuyor.")

    from huggingface_hub import snapshot_download
    model_dir = WORKSPACE / "qwen_model"
    snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION, local_dir=model_dir)

    if str(source_dir) not in sys.path:
        sys.path.insert(0, str(source_dir))
    from src.models.qwen3_vl_embedding import Qwen3VLEmbedder

    capability = torch.cuda.get_device_capability()
    dtype = torch.float16 if capability[0] < 8 else torch.bfloat16
    embedder = Qwen3VLEmbedder(
        model_name_or_path=str(model_dir),
        fps=1.0, max_frames=16, max_length=16384,
        torch_dtype=dtype, attn_implementation=ATTN_IMPL,
    )

    from app.ingestion.generic_loader import iter_window_records, release_frames
    from app.mrl import truncate_and_normalize

    output_root = WORKSPACE / "embedding_bundle"
    shard_root = output_root / "shards"
    shard_root.mkdir(parents=True, exist_ok=True)

    def atomic_save_npy(path, array):
        temp = path.with_name(path.name + ".partial")
        with temp.open("wb") as handle:
            np.save(handle, array)
        temp.replace(path)

    def completed_ids():
        ids = set()
        for path in shard_root.glob("shard_*.json"):
            try:
                payload = json.loads(path.read_text(encoding="utf-8"))
                if payload.get("status") == "completed":
                    ids.add(int(payload["shard_id"]))
            except Exception:
                pass
        return ids

    def write_shard(index, records):
        segment_rows, base_vectors = [], []
        try:
            for start in range(0, len(records), EMBED_BATCH_SIZE):
                batch = records[start:start+EMBED_BATCH_SIZE]
                result = embedder.process([{"video": record.frames} for record in batch])
                vectors = result.detach().cpu().float().numpy().astype(np.float32, copy=False)
                if vectors.shape != (len(batch), 2048) or not np.isfinite(vectors).all():
                    raise RuntimeError(f"Geçersiz Qwen batch: {vectors.shape}")
                for record, vector in zip(batch, vectors, strict=True):
                    base_vectors.append(truncate_and_normalize(vector, 2048))
                    segment_rows.append({
                        "segment_id": record.segment_id,
                        "dataset_id": record.dataset_id,
                        "video_id": record.video_id,
                        "t_start": record.t_start,
                        "t_end": record.t_end,
                        "chunk_index": record.chunk_index,
                        "source_path": record.metadata["source_path"],
                        "telemetry_json": json.dumps(record.telemetry, ensure_ascii=False),
                        "extra_json": json.dumps(record.extra, ensure_ascii=False),
                    })
        finally:
            release_frames(records)

        base = np.stack(base_vectors).astype(np.float32)
        dim_files = {}
        for dim in MRL_DIMENSIONS:
            projected = np.stack([truncate_and_normalize(v, dim) for v in base])
            path = shard_root / f"shard_{index:05d}_dim{dim}.npy"
            atomic_save_npy(path, projected)
            dim_files[str(dim)] = path.name

        table_path = shard_root / f"shard_{index:05d}_segments.parquet"
        temp_table = table_path.with_name(table_path.name + ".partial")
        pd.DataFrame(segment_rows).to_parquet(temp_table, index=False)
        temp_table.replace(table_path)

        payload = {
            "status":"completed", "shard_id":index, "vector_count":len(segment_rows),
            "dimensions":list(MRL_DIMENSIONS), "dimension_files":dim_files,
            "segments_file":table_path.name, "manifest_hash":parsed_manifest.manifest_hash,
            "model_id":MODEL_ID, "model_revision":MODEL_REVISION, "source_commit":SOURCE_COMMIT,
        }
        meta = shard_root / f"shard_{index:05d}.json"
        temp_meta = meta.with_name(meta.name + ".partial")
        temp_meta.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        temp_meta.replace(meta)
        return payload

    completed = completed_ids()
    records = iter_window_records(parsed_manifest, data_root=DATA_ROOT, configured=configured)
    buffer, total = [], 0
    for record in records:
        if MAX_WINDOWS is not None and total >= MAX_WINDOWS:
            release_frames([record])
            break
        shard_id = total // SHARD_WINDOWS
        if shard_id in completed:
            release_frames([record])
            total += 1
            continue
        buffer.append(record)
        total += 1
        if len(buffer) >= SHARD_WINDOWS:
            print(write_shard(shard_id, buffer))
            buffer = []
    if buffer:
        print(write_shard((total-1)//SHARD_WINDOWS, buffer))

    shard_meta = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(shard_root.glob("shard_*.json"))]
    bundle_manifest = {
        "schema_version":1, "generated_at_utc":datetime.now(timezone.utc).isoformat(),
        "dataset_id":DATASET_ID, "dataset_manifest_hash":parsed_manifest.manifest_hash,
        "model_id":MODEL_ID, "model_revision":MODEL_REVISION,
        "source_commit":SOURCE_COMMIT, "dimensions":list(MRL_DIMENSIONS), "shards":shard_meta,
    }
    (output_root / "manifest.json").write_text(json.dumps(bundle_manifest, indent=2), encoding="utf-8")
    print("Embedding bundle:", output_root)

## 8C. Aynı hosttaki Docker FAZ11 sistemine ingest

Yalnız `MODE == "docker_ingest"` olduğunda çalışır. Colab'da reddedilir. Güvenlik için servis başlatma ve ingest bayrakları varsayılan olarak kapalıdır.

In [ ]:
if MODE != "docker_ingest":
    print("Atlandı: MODE docker_ingest değil.")
else:
    if IS_COLAB:
        raise RuntimeError("docker_ingest Colab'da desteklenmez; kurum hostunda local Jupyter kullanın.")

    env_file = (REPO_ROOT / ENV_FILE).resolve()
    if not env_file.is_file():
        raise FileNotFoundError(env_file)

    container_manifest = f"/workspace/datasets/generated/{MANIFEST_PATH.name}"
    run([sys.executable, REPO_ROOT/"scripts"/"preflight.py",
         "--dataset", MANIFEST_PATH, "--env-file", env_file,
         "--json-out", REPO_ROOT/"artifacts"/"faz11"/"preflight.json"], cwd=REPO_ROOT)

    if START_DOCKER_SERVICES:
        run(["docker","compose","--env-file",env_file,
             "-f",REPO_ROOT/"docker-compose.yml","-f",REPO_ROOT/"docker-compose.gpu.yml",
             "up","-d","--build"], cwd=REPO_ROOT)
        run(["docker","compose","--env-file",env_file,"ps"], cwd=REPO_ROOT)
    else:
        print("START_DOCKER_SERVICES=False: servis başlatma atlandı.")

    if RUN_DOCKER_INGEST:
        run(["docker","compose","--env-file",env_file,"exec","api",
             "python","-m","app.ingestion.ingest",
             "--dataset",container_manifest,"--resume"], cwd=REPO_ROOT)
    else:
        print("RUN_DOCKER_INGEST=False: ingest atlandı.")
        print("Hazır komut:")
        print(f"docker compose --env-file {env_file} exec api "
              f"python -m app.ingestion.ingest --dataset {container_manifest} --resume")

    print("UI: http://127.0.0.1:7860")
    print("API: http://127.0.0.1:8000/docs")

## 9. Oturum özeti

In [ ]:
summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "mode": MODE, "data_source": DATA_SOURCE, "data_root": str(DATA_ROOT),
    "dataset_id": DATASET_ID, "video_count": len(video_paths),
    "telemetry_csv_count": len(csv_paths), "manifest_path": str(MANIFEST_PATH),
    "manifest_hash": parsed_manifest.manifest_hash,
    "preflight_status": preflight_report["status"],
    "preflight_path": str(PREFLIGHT_PATH),
    "onboarding_bundle": archive_path,
    "embedding_bundle": str(WORKSPACE/"embedding_bundle") if MODE=="portable_embedding" else None,
    "ui_url": "http://127.0.0.1:7860" if MODE=="docker_ingest" else None,
}
summary_path = WORKSPACE / "session_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Session summary:", summary_path)